### Group Members:

- Name, matriculation number
- Name, matriculation number
- Name, matriculation number

# Assignment 5 - Time Series Processing

The goal of this assignment is to explore time series processing for **sleep stage classification**, using models that capture **temporal dynamics** in EEG recordings. 
We will implement and compare different neural network architectures **— 1D Convolution, Recurrent Neural Network (RNN), Long Short-Term Memory (LSTM), and Attention-based network —** on a subset of the **Sleep-EDF dataset**. 

The dataset contains EEG signals recorded from human subjects at the Fpz-Cz electrode location on the scalp. 
The EEG signal is sampled at **100 Hz**, meaning **each second contains 100 data points**. 
As is common practice in sleep research, the continuous signal is segmented into **non-overlapping 30-second windows**, resulting in **3000 sequential samples per segment**. 
In other words, each segment represents **a single-channel time series** reflecting brain activity during that **30-second interval**.
Each EEG segment is labeled with one of the following five sleep stages: Wake, N1 (Light Sleep), N2 (Stable Sleep), N3 (Deep Sleep), or REM (Rapid Eye Movement), **corresponding to labels 0 through 4**. 

Each individual's data is stored in a separate `.npz` file. 
For this assignment, the dataset will be split by subject: a portion of individuals will be used for training, another subset for validation, and the remaining subjects for testing.

**1D convolution** is well-suited for time-series data like EEG signals, as it applies localized filters along the time axis to automatically capture temporal patterns such as trends, transitions, and periodicities, while preserving sequence order. 
Stacking **Conv1d** and pooling layers helps compress the sequence length efficiently without losing important information, enabling faster and more scalable processing. 
Therefore, in this assignment, to handle the long input sequences (**3000 time steps**), you are required to design **a small 1D convolutional neural network** as a feature extractor, which will also be utilized by all subsequent models, including the RNN, LSTM, and Attention-based network. 
Each model will then process the reduced sequence to classify the sample into one of the sleep stages.

The entire assignment will be implemented using `PyTorch`.

**Structure of Networks**

1. **Feature Extractor**: *The designed feature extractor network based on Conv1d operations*

2. **Classifier Networks:**

   a) **Pooling**: *Feature extractor* ⟶  *Global pooling* ⟶ *Classifier*

   b) **RNN**: *Feature extractor* ⟶  *RNN* ⟶ *Classifier*

   c) **LSTM**: *Feature extractor* ⟶  *LSTM* ⟶ *Classifier*

   d) **Attention with class token**: *Feature extractor* ⟶  *Self-attention with Class token* ⟶ *Class token* ⟶ *Classifier*

**Class Imbalance**

After training all networks, we will evaluate and compare their performance. 
The best-performing model will then be further refined to better address class imbalance by applying two strategies:

* **Global Class Weighting**: Apply a weighted loss function where each class is assigned a weight inversely proportional to its frequency in the entire training set. 
This ensures that samples from underrepresented classes contribute more significantly to the loss.

* **DynaMit Approach**: Implement a dynamic class weighting method, where class weights are computed per batch during training based on the class distribution within each batch. 
Unlike fixed global class weights, this approach adapts dynamically to the composition of each batch.

The final results from each model will be compared based on overall classification accuracy, balanced accuracy and per-class performance.

## Theoretical Sections

In this section, we address following fundamental aspects:  

1. Design the feature extractor network for time series processing
2. Compute the output feature map size of the designed feature extractor network
3. Discuss pros and cons of other feature extraction methods 
4. Explore normalization methods for EEG time series signals
5. Discuss alternative attention methods for EEG classification
6. Handle variable-length sequences

#### Task 1.1: 1D Convolutional Network

Design a **feature extractor** that **must** include **four 1D convolutional layers** with $Q_1$, $Q_2$, $Q_3$, and $Q_4$ output channels, respectively. 
Explain the reasoning why you have designed the network in the way you did.

**Note:** You are encouraged to add additional layers such as `MaxPool1D`, `BatchNorm1D`, `non-linear activations` (e.g., ReLU, GELU, etc.) and `Dropout` wherever you find appropriate, in order to improve feature compression and model generalization.
Also, make use of padding, striding or any other techniques in your convolution and pooling operations.
A reasonable number of output channels could be $Q_4=128$.

Answer:

**Layer 1**:
- 1D convolutional layer with $Q_1 = 64$ output channels, kernel size $51$, stride $6$, and padding $25$.

- Batch normalization over $Q_1 = 64$ channels. 

- GELU activation function.  

- 1D max pooling layer with kernel size $4$, stride $2$, padding $1$.  

- Dropout layer with dropout probability $0.5$. 

**Layer 2**:
- 1D convolutional layer with $Q_2 = 64$ output channels, kernel size $9$, stride $2$, and padding $4$.

- Batch normalization over $Q_2 = 64$ channels.  

- GELU activation function. 

**Layer 3**:
- 1D convolutional layer with $Q_3 = 128$ output channels, kernel size $7$, stride $1$, and padding $3$.

- Batch normalization over $Q_3 = 128$ channels.  

- GELU activation function. 

- 1D max pooling layer with kernel size $2$, stride $2$, padding $1$.  

**Layer 4**:
- 1D convolutional layer with $Q_4 = 128$ output channels (final feature dimension), kernel size $5$, stride $1$, and padding $2$.

- Batch normalization over $Q_4 = 128$ channels.  

- GELU activation function.  


#### Task 1.2: Calculate Feature Map Dimension

Given the network designed in Task 1.1, calculate the dimensionality of the extracted feature map, including the new sequence length, the number of channels and the batch size.

**Hint:** Consider how each `Conv1d` and `MaxPool1d` layer affects the time dimension.

Answer:

Only **Conv1d** and **MaxPool1d** layers affect the sequence length. Activation functions, batch normalization, and dropout do **not** change it.

The output length $L_{\text{out}}$ of a 1D convolution or pooling layer is given by:

$$
L_{\text{out}} = \left\lfloor \frac{L_{\text{in}} + 2 \cdot p - U}{S} + 1 \right\rfloor
$$

Where:
- $L_{\text{in}}$ is the input sequence length,
- $U$ is the kernel size,
- $S$ is the stride,
- $p$ is the padding.

---

**Layer 1: Conv1d**

- Kernel size: $51$, stride: $6$, padding: $25$
$$
L_1 = \left\lfloor \frac{3000 + 2 \cdot 25 - 51}{6} \right\rfloor + 1 = \left\lfloor \frac{3000-1}{6} \right\rfloor + 1 = 499 + 1 = 500
$$

**Layer 1: MaxPool1d**

- Kernel size: $4$, stride: $2$, padding: $1$
$$
L_2 = \left\lfloor \frac{500 + 2 \cdot 1 - 4}{2} \right\rfloor + 1 = \left\lfloor \frac{500-2}{2} \right\rfloor + 1 = 250
$$

---

**Layer 2: Conv1d**

- Kernel size: $9$, stride: $2$, padding: $4$
$$
L_3 = \left\lfloor \frac{250 + 2 \cdot 4 - 9}{2} \right\rfloor + 1 = 125
$$

---

**Layer 3: Conv1d**

- Kernel size: $7$, stride: $1$, padding: $3$
$$
L_4 = \left\lfloor \frac{125 + 2 \cdot 3 - 7}{1} \right\rfloor + 1 = 125
$$

**Layer 3: MaxPool1d**

- Kernel size: $2$, stride: $2$, padding: $1$
$$
L_6 = \left\lfloor \frac{125 + 2 \cdot 1 - 2}{2} \right\rfloor + 1 = 63
$$

---

**Layer 4: Conv1d**

- Kernel size: $7$, stride: $1$, padding: $3$
$$
L_5 = \left\lfloor \frac{63 + 2 \cdot 3 - 7}{1} \right\rfloor + 1 = 63
$$

---

**Final size of features: $B\times Q_4\times 63$**

#### Task 1.3: Feature Extractors

When performing classification on brain waves, we can apply temporal models such as LSTM, GRU, or attention-based networks for time-series EEG signals.
However, high-frequency wave structures are difficult to learn from.
Therefore, in the literature there exist several ways of how to handle such high-frequency data.

**Which methods or architectures can be used for feature extraction?** 
Please explain possible methods and their advantages/disadvantages.

Answer:

- **Stacked 1D Convolutional Layers (Conv1d):** Apply temporal filters to capture temporal patterns and reduce sequence length while retaining important features. Commonly used as the first step before recurrent or attention layers. 
  + Advantage: Can learn task-specific filters from data
  + Disadvantage: Needs large amounts of data for training, and proper design of the network

- **Statistical Feature Extraction:** Compute features like mean, variance, skewness, or kurtosis over fixed-size sliding windows to summarize local signal behavior. Useful for reducing raw signal to compact descriptors.
  + Advantage: Captures well-known and well-understood fixed features from dynamic sequences
  + Disadvantage: Removes fine details and temporal coherence

- **Frequency domain transformations** such as **Short-Time Fourier Transform (STFT)** or **wavelet transform**: Convert the time-domain signal into time-frequency representations to capture oscillatory activity. This helps models learn from the spectral content, especially relevant for EEG.
  + Advantage: Keeps temporal information without requiring training
  + Disadvantage: Introduces more hyperparameters (number of frequency bands, frequency ranges) that need to be tuned

- **Autoencoders / 1D Convolutional Autoencoders:** Learn compressed, denoised representations in an unsupervised way. These can be used to reduce dimensionality or pretrain feature encoders. 
  + Advantage: Can be pre-trained on huge amounts of *unlabeled* publicly-available data
  + Disadvantage: Might overfit to class biased in the unlabeled data

- **Band-Pass Filtering / EEG Frequency Decomposition:** Extract frequency-specific signals (e.g., delta, theta, alpha, beta, gamma bands) known to correlate with brain states. Helps isolate physiologically meaningful patterns before modeling.
  + Advantage: Domain-specific knowledge can be incorporated to extract frequencies that are known to contribute
  + Disadvantage: Requires domain-specific knowledge and ignores information in other frequency bands

#### Task 1.4: Normalization Methods

Input normalization is an important preprocessing step that helps in providing inputs in reasonable input ranges.
Especially for such sequence-based systems, there exist several variations of input normalization.

**How could we normalize the time sequence brain signals (e.g., EEG) for sleep stage classification?** 
Please explain possible methods.

Answer:

EEG signals vary in amplitude across channels, time, and subjects. Proper normalization can improve model stability and performance.

- **Z-score normalization**, applied either **globally across the entire dataset** or **per sequence**, involves subtracting the mean and dividing by the standard deviation computed across all subjects or within individual sequences. This method centers and scales the data based on Gaussian assumptions.

- **Min-max normalization**, applied either **globally across the entire dataset** or **per sequence**, rescales the signal values to a fixed range such as $[0, 1]$ based on the minimum and maximum values observed.

- **Max-abs normalization**, applied either **globally across the entire dataset** or **per sequence**, rescales the signal values to a fixed range such as $[-1, 1]$ based on the maximum absolute values observed.

**Note:** While all three strategies are mathematically valid, **min-max and max-abs normalization applied globally or per sequence** are generally discouraged for EEG signals. EEG recordings are often noisy and prone to outliers (e.g., muscle artifacts, blinks), which can distort the global minimum and maximum, leading to overly compressed or skewed feature scales. In contrast, **z-score normalization is more robust to outliers** — the standard deviation is less sensitive to extreme values than the min/max range. So, even if there are spikes in the EEG, **global z-score normalization does not get distorted as severely**.

- Z-score normalization **per channel per subject**, i.e., subtract mean and divide by standard deviation computed from each subject's own recordings. 
> **Subject variability:** EEG baselines vary between subjects. Global scaling ignores individual differences in amplitude. This approach accounts for **subject-specific amplitude baselines**, which global normalization may ignore. 

#### Task 1.5: Attention Mechanisms

In Task 2.7, we will make use of standard self-attention with a class token to extract sequence-level information to perform the final classification task.
This includes concatenating a learnable class token to the features extracted by the feature extractor, apply self-attention, and then evaluate the result of the transformed class token.
While this is the most commonly applied way of extracting sequence-level information in attention-based networks, other ways of extracting such sequence-level information also exist.

**Explain possible methods for using attention mechanisms -- not only self-attention -- to extract sequence-level information.** 

Answer:

In sleep stage classification using EEG, various attention mechanisms can help the model focus on meaningful temporal and spatial features beyond what standard self-attention provides:

- **Attention:**  
  Unlike self-attention, where each time step attends to all other time steps (i.e., pairwise interactions across the full sequence), attention typically focuses on identifying which time steps are most relevant or has most important feature for the task at hand. Instead of computing interactions between every pair of time steps, **a learnable class token** serves as a summary representation. During attention, only this class token attends to all time steps — extracting information from the sequence into a single global representation. The remaining tokens do not attend to each other, so their pairwise interactions are skipped entirely.

  This design avoids unnecessary computation by removing the full attention map between all time steps. It also does not require adding a class token as input. Since the final decision often depends only on the class token (e.g., in classification), focusing attention from all time steps into one token is both efficient and task-aligned. 

- **Local Attention:**  
  Instead of computing attention across the entire sequence, local attention restricts the model to attend within a limited temporal window. This is computationally efficient and aligns well with the fact that many sleep stage features are locally encoded.

- **Hierarchical Attention:**  
  EEG patterns often have a nested structure—brief waveforms (like spindles) occur within longer epochs. Hierarchical attention allows the model to first extract features from short windows and then attend across windows, capturing both fine and coarse temporal structures.

- **Channel-wise Attention:**  
  In multi-channel EEG data, this mechanism learns to weigh different EEG electrodes (channels) according to their relevance. It helps the model focus on the most informative brain regions for each sleep stage, adapting to inter-subject variability. However, in our data we have only one channel, so this does not apply for our example.

#### Task 1.6: Variable Sequence Lengths

In our small example above, we ensured that the dataset always contained the same sequence lengths for all samples.
However, in many tasks, including natural language processing, sequence lengths can differ between samples.

**Explain how we can handle situations where the input sequences have different lengths?**
Provide at least two different strategies.

Answer: 

**Common Strategies to Handle Different Input Sizes**

**1. Padding:**  
Shorter sequences are padded with a special value (e.g., 0 or `<pad>`) to match the length of the longest sequence in the batch. 
This allows efficient batch processing on GPUs. 
Padding ensures uniform tensor shapes, but since it introduces non-informative values, it must be handled carefully during training.

**2. Masking:**  
Masking is used to prevent the model from learning from padded values. 
A binary mask is created to indicate valid time steps (marked with 1) and padded positions (marked with 0). 
This mask is applied during loss calculation to ignore padded parts of the sequence. 

In attention-based models, masking is also used during the attention score computation to ensure the model does not attend to padded tokens. 
Specifically, the attention weights for padded positions are set to very negative values (e.g., $-\infty$) before the softmax operation, effectively excluding them from contributing to the output.

**3. Sequence Bucketing:**  
Sequences of similar lengths are grouped into the same batch. 
This reduces the amount of padding needed within each batch, leading to improved computational efficiency and more stable training. 
Bucketing is especially helpful in NLP and speech processing tasks, where sequence lengths can vary significantly.

**4. Truncation:**  
Long sequences are clipped to a predefined maximum length to save memory and computation. 
This is common in real-time applications or classification tasks where only the beginning of the sequence carries important information. 
However, truncation may result in the loss of valuable temporal context if not applied carefully.

**5. Sliding Windows:**  
The time series is divided into fixed-size overlapping or non-overlapping windows, which are processed independently. 
This approach is useful in event detection, physiological signal analysis, and local pattern recognition. 
Sliding windows allow models to work with a consistent input size without relying heavily on padding or truncation, but the outputs of various windows might need to be combined into final predictions.